# Processing of a resting-state EEG dataset for sleep deprivation

Bad channels are interpolated

## PRE PROCESSING

#### The intention is to provide epochs(60 seconds) to the machine and try predicting if the case is sleep deprivation or normal sleep.

In [257]:
from matplotlib import pyplot as plt
import os
import pandas as pd
import numpy as np
from scipy import signal
import mne
import mne_icalabel
from mne.preprocessing import ICA
from autoreject import Ransac  
from mne_icalabel import label_components
%matplotlib inline

In [258]:
# lists to collect data
X_combained_train_list = []
y_combained_train_list = []
X_combained_test_list = []
y_combained_test_list = []


In [259]:
n_epochs = 0
for i in range(1, 72):
    if i == 28 or i== 1 or i== 44 or i==39 or i==43 or i  in [7,8,11,13,18,24,29,31,35,36,37,38,40,47,48,50,54,58,59,63,67,70,71]: #46
        continue  # Skip participants due to missing data
    for j in range(1, 3):
        
        #Define file path
        file_path = f'D:\\BCI\\TASKS\\Task-8\\ds004902_data\\sub-{i:02d}\\ses-{j}\\eeg\\sub-{i:02d}_ses-{j}_task-eyesopen_eeg.set'#The d indicates that the value is a decimal integer, while the 02 specifies that the output should be padded with leading zeros if necessary to reach a width of two characters.
        epochs_file = f'D:\\BCI\\TASKS\\Task-8\\processed_epochs_60\\sub-{i:02d}_ses-{j}_epochs.fif'  # Path to save/load processed epochs

        
        eeg_channels = ["Fp1", "AF3", "AF7", "Fz", "F1", "F3", "F5", "F7","FC1", "FC3", "FC5", "FT7","Cz", "C1", "C3", "C5", "T7","CP1", "CP3", "CP5", "TP7", "TP9","Pz", "P1", "P3", "P5", "P7","PO3", "PO7", "Oz", "O1","Fpz", "Fp2", "AF4", "AF8","F2", "F4", "F6", "F8","FC2", "FC4", "FC6", "FT8","C2", "C4", "C6", "T8","CPz", "CP2", "CP4", "CP6","TP8", "TP10","P2", "P4", "P6", "P8","POz", "PO4", "PO8", "O2"]
               
        if os.path.exists(epochs_file):
            print(f"Loading processed epochs from {epochs_file}")
            epochs = mne.read_epochs(epochs_file, preload=True)
        else:
            print(f"\nProcessing: {file_path}\n")

            #Define EEG channels
            eeg_channels = ["Fp1", "AF3", "AF7", "Fz", "F1", "F3", "F5", "F7","FC1", "FC3", "FC5", "FT7","Cz", "C1", "C3", "C5", "T7","CP1", "CP3", "CP5", "TP7", "TP9","Pz", "P1", "P3", "P5", "P7","PO3", "PO7", "Oz", "O1","Fpz", "Fp2", "AF4", "AF8","F2", "F4", "F6", "F8","FC2", "FC4", "FC6", "FT8","C2", "C4", "C6", "T8","CPz", "CP2", "CP4", "CP6","TP8", "TP10","P2", "P4", "P6", "P8","POz", "PO4", "PO8", "O2"]
            
            #Load EEG data
            raw = mne.io.read_raw_eeglab(file_path, preload=True)
            eeg_raw=raw.pick(mne.pick_channels(raw.info['ch_names'], include=eeg_channels))
            
            #Montage setting
            montage = mne.channels.make_standard_montage('standard_1020')
            eeg_raw.set_montage(montage)

            #Filtering
            eeg_raw.filter(1., 40.)#, fir_design='firwin')

            #Average Referencing
            eeg_raw.set_eeg_reference('average', projection=True)
            eeg_raw.apply_proj()
            
            #EPOCHING
            epochs = mne.make_fixed_length_epochs(eeg_raw, duration=60.0, overlap=0.0,preload=True)
            
            #RANSAC for bad channel detection
            ransac = Ransac(verbose=False, n_jobs=1)
            ransac.fit(epochs)
            raw.info['bads'].extend(ransac.bad_chs_)   

            #interpolate bad channels
            raw.interpolate_bads(reset_bads=True)

            
            #ICA for artifact removal
            ica = mne.preprocessing.ICA(n_components=20, method='infomax', fit_params=dict(extended=True), random_state=42, max_iter=1000)
            ica.fit(epochs)

            ic_labels = label_components(epochs, ica, method='iclabel')
            exclude_idx = []
            for idx, label in enumerate(ic_labels["labels"]):
                if label not in ["brain","other","muscle artifact"] and ic_labels["y_pred_proba"][idx] > 0.5:
                    exclude_idx.append(idx)

            ica.apply(epochs,exclude= exclude_idx)        

            # Save processed epochs
            os.makedirs(os.path.dirname(epochs_file), exist_ok=True) #creates the necessary directories for the file path if they don't exist.
            epochs.save(epochs_file, overwrite=True)
            print(f"Saved processed epochs to {epochs_file}")

        if j==1:
            labels = 0
        elif j==2:
            labels = 1
        labels = pd.DataFrame({'label': [labels]*len(epochs)})

        # Extract data and labels
        data = epochs.get_data()  # shape: (n_epochs, n_channels, n_times)

        # 1. Frontal & Temporal: 
        ft_eeg_channels = [ch for ch in eeg_channels if ch.startswith(('C', 'P', 'CP', 'PO')) ]#and not ch.startswith(('CP'))] 
        data_ft = epochs.copy().pick(ft_eeg_channels)

        # 2. Temporal: 
        temporal_eeg_channels = [ch for ch in eeg_channels if ch.startswith(('T', 'FT', 'TP'))]
        data_temporal = epochs.copy().pick(temporal_eeg_channels)

        # 3. Parietal: 
        parietal_eeg_channels = [ch for ch in eeg_channels if ch.startswith(('P', 'CP', 'PO'))]
        data_parietal = epochs.copy().pick(parietal_eeg_channels)

        # 4. Occipital:
        occipital_eeg_channels = [ch for ch in eeg_channels if ch.startswith('O')]
        data_occipital = epochs.copy().pick(occipital_eeg_channels)

        # 5. Central channels:
        central_channels = [ch for ch in eeg_channels if ch.startswith('C') and not ch.startswith('CP')]
        data_central = epochs.copy().pick(central_channels)              

        #if (i == 39 or i == 43) and j == 2:
        #    data = signal.decimate(data, 10, axis=2) # Downsample by a factor of 10 to match time points.

        epoch_labels = labels['label'].values  
        
        if i <= 56:  # Training data
            X_combained_train_list.append(data_ft.get_data())
            
            y_combained_train_list.append(epoch_labels)
            
        else:  # Testing data
            X_combained_test_list.append(data_ft.get_data())
            
            y_combained_test_list.append(epoch_labels)
         

# After the loop, concatenate into arrays
X_combained_train = np.concatenate(X_combained_train_list, axis=0)


y_combained_train = np.concatenate(y_combained_train_list, axis=0)


X_combained_test = np.concatenate(X_combained_test_list, axis=0)


y_combained_test = np.concatenate(y_combained_test_list, axis=0)


print("Training and testing data processing complete.")

Loading processed epochs from D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-1_epochs.fif
Reading D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-1_epochs.fif ...
    Found the data of interest:
        t =       0.00 ...   59998.00 ms
        0 CTF compensation matrices available
Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated


C:\Users\suvan\AppData\Local\Temp\ipykernel_19092\3871019675.py:16: RuntimeWarning: This filename (D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-1_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(epochs_file, preload=True)


Loading processed epochs from D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-2_epochs.fif
Reading D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-2_epochs.fif ...
    Found the data of interest:
        t =       0.00 ...   59998.00 ms
        0 CTF compensation matrices available


C:\Users\suvan\AppData\Local\Temp\ipykernel_19092\3871019675.py:16: RuntimeWarning: This filename (D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-2_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(epochs_file, preload=True)


Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated
Loading processed epochs from D:\BCI\TASKS\Task-8\processed_epochs_60\sub-03_ses-1_epochs.fif
Reading D:\BCI\TASKS\Task-8\processed_epochs_60\sub-03_ses-1_epochs.fif ...
    Found the data of interest:
        t =       0.00 ...   59998.00 ms
        0 CTF compensation matrices available
Not setting metadata
5 matching events found


...

Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated
Loading processed epochs from D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-1_epochs.fif
Reading D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-1_epochs.fif ...
    Found the data of interest:
        t =       0.00 ...   59998.00 ms
        0 CTF compensation matrices available
Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated


C:\Users\suvan\AppData\Local\Temp\ipykernel_19092\3871019675.py:16: RuntimeWarning: This filename (D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-1_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(epochs_file, preload=True)


Loading processed epochs from D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-2_epochs.fif
Reading D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-2_epochs.fif ...
    Found the data of interest:
        t =       0.00 ...   59998.00 ms
        0 CTF compensation matrices available


C:\Users\suvan\AppData\Local\Temp\ipykernel_19092\3871019675.py:16: RuntimeWarning: This filename (D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-2_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(epochs_file, preload=True)


Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated
Training and testing data processing complete.


## FEATURE EXTRACTION 

In [260]:
from scipy import stats

In [261]:
#Time-domain feature extraction functions
def mean(sample_data):
    return np.mean(sample_data, axis=-1)

def std(sample_data):
    return np.std(sample_data, axis=-1)

def zscore(sample_data):
    return stats.zscore(sample_data, axis=-1)

def ptp(sample_data):
    return np.ptp(sample_data, axis=-1)

def min(sample_data):
    return np.min(sample_data, axis=-1)

def max(sample_data):
    return np.max(sample_data, axis=-1)

def var(sample_data):
    return np.var(sample_data, axis=-1)

def rms(sample_data):
    return np.sqrt(np.mean(sample_data**2, axis=-1))

def skewness(sample_data):
    return stats.skew(sample_data, axis=-1)   

def kurtosis(sample_data):
    return stats.kurtosis(sample_data, axis=-1)


def mean_square(sample_data):
    return np.mean(sample_data**2, axis=-1)

def hjorth_params(sample_data):
    first_deriv = np.diff(sample_data, axis=-1)
    second_deriv = np.diff(first_deriv, axis=-1)

    var_zero = var(sample_data)
    var_d1 = var(first_deriv)
    var_d2 = var(second_deriv)
    
    activity = var_zero
    mobility = np.sqrt(var_d1 / var_zero)
    complexity = np.sqrt(var_d2 / var_d1) / mobility
    
    return activity, mobility, complexity

In [262]:
# Frequency-domain feature extraction functions
def bandpower_delta(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    delta_band = (0.5, 4)
    mask = (freqs >= delta_band[0]) & (freqs <= delta_band[1])
    delta_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    return delta_power

def bandpower_theta(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    theta_band = (4, 8)
    mask = (freqs >= theta_band[0]) & (freqs <= theta_band[1])
    theta_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    return theta_power

def bandpower_alpha(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    alpha_band = (8, 13)
    mask = (freqs >= alpha_band[0]) & (freqs <= alpha_band[1])
    alpha_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    return alpha_power  

def bandpower_beta(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    beta_band = (13, 30)
    mask = (freqs >= beta_band[0]) & (freqs <= beta_band[1])
    beta_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    return beta_power       

def bandpower_gamma(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    gamma_band = (30, 40)
    mask = (freqs >= gamma_band[0]) & (freqs <= gamma_band[1])
    gamma_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    return gamma_power

def spectral_entropy(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    psd_norm = psd_values / np.sum(psd_values, axis=axis, keepdims=True)
    spectral_entropy = -np.sum(psd_norm * np.log2(psd_norm + 1e-10), axis=axis)
    return spectral_entropy

In [263]:
def feature_concatination(sample_data):
    feature_vector = []
    
    # Time-domain features
    #feature_vector.append(mean(sample_data))
    #feature_vector.append(std(sample_data))
    #feature_vector.append(ptp(sample_data))
    #feature_vector.append(min(sample_data))
    #feature_vector.append(max(sample_data))
    #feature_vector.append(var(sample_data))
    #feature_vector.append(rms(sample_data))
    #feature_vector.append(skewness(sample_data))
    #feature_vector.append(kurtosis(sample_data))
    #activity, mobility, complexity = hjorth_params(sample_data)
    #feature_vector.append(activity)
    #feature_vector.append(mobility)
    #feature_vector.append(complexity)

    # Frequency-domain features
    feature_vector.append(bandpower_delta(sample_data))
    feature_vector.append(bandpower_theta(sample_data))
    feature_vector.append(bandpower_alpha(sample_data))
    feature_vector.append(bandpower_beta(sample_data))
    feature_vector.append(bandpower_gamma(sample_data))
    feature_vector.append(spectral_entropy(sample_data))
    
    return np.concatenate(feature_vector, axis=-1)

In [264]:
from tqdm import tqdm_notebook
X_combained_train_features=[]
for datta in tqdm_notebook(X_combained_train):
    features = feature_concatination(datta)
    X_combained_train_features.append(features)

X_combained_train_features=np.array(X_combained_train_features)
X_combained_train_features.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19092\2874730755.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_combained_train):


  0%|          | 0/339 [00:00<?, ?it/s]

(339, 168)

In [265]:
from tqdm import tqdm_notebook
X_combained_test_features=[]
for datta in tqdm_notebook(X_combained_test):
    features = feature_concatination(datta)
    X_combained_test_features.append(features)

X_combained_test_features=np.array(X_combained_test_features)
X_combained_test_features.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19092\3846018410.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_combained_test):


  0%|          | 0/85 [00:00<?, ?it/s]

(85, 168)

## MACHINE LEARNING

### Combained

In [266]:
from sklearn.ensemble import RandomForestClassifier

In [267]:
X_combained_train_features.shape, y_combained_train.shape

((339, 168), (339,))

In [268]:
# Instantiate model with 1000 decision trees
rf = RandomForestClassifier(n_estimators = 40, random_state = 42)
# Train the model on training data
rf.fit(X_combained_train_features, y_combained_train)

RandomForestClassifier(n_estimators=40, random_state=42)

In [269]:
predictions = rf.predict(X_combained_test_features)

In [270]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, classification_report
cm = confusion_matrix(y_combained_test, predictions)
print("Confusion Matrix:\n", cm)
print("\n")
cf = classification_report(y_combained_test, predictions)
print("Classification Report:\n", cf)
print("\n")
accuracy = accuracy_score(y_combained_test, predictions)
print("Accuracy:", accuracy)

Confusion Matrix:
 [[21 23]
 [ 9 32]]


Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.48      0.57        44
           1       0.58      0.78      0.67        41

    accuracy                           0.62        85
   macro avg       0.64      0.63      0.62        85
weighted avg       0.64      0.62      0.62        85



Accuracy: 0.6235294117647059


In [271]:
from sklearn.svm import SVC

In [272]:
model = SVC(C=1, kernel='rbf', gamma='scale', random_state=42) #default parameters
model.fit(X_combained_train_features, y_combained_train)

SVC(C=1, random_state=42)

In [273]:
predictions = model.predict(X_combained_test_features)

In [274]:
cm = confusion_matrix(y_combained_test, predictions)
print("Confusion Matrix:\n", cm)
print("\n")
cf = classification_report(y_combained_test, predictions)
print("Classification Report:\n", cf)
print("\n")
accuracy = accuracy_score(y_combained_test, predictions)
print("Accuracy:", accuracy)

Confusion Matrix:
 [[44  0]
 [41  0]]


Classification Report:
               precision    recall  f1-score   support

           0       0.52      1.00      0.68        44
           1       0.00      0.00      0.00        41

    accuracy                           0.52        85
   macro avg       0.26      0.50      0.34        85
weighted avg       0.27      0.52      0.35        85



Accuracy: 0.5176470588235295


c:\Users\suvan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\suvan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\suvan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [275]:
#Grid Search CV for SVM hyperparameter tuning
from sklearn.model_selection import GridSearchCV

In [276]:
param_grid = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf', 'poly', 'sigmoid'], 'gamma': ['scale', 'auto']}

In [277]:
#grid= GridSearchCV(model, param_grid, refit=True, verbose=3)
#grid.fit(X_train_features, y_train)

In [278]:
#grid.best_params_

In [279]:
#grid.best_estimator_

In [280]:
#grid_predictions = grid.predict(X_test_features)

In [281]:
'''cm = confusion_matrix(y_test, grid_predictions)
print("Confusion Matrix:\n", cm)
print("\n")
cf = classification_report(y_test, grid_predictions)
print("Classification Report:\n", cf)   
print("\n")
accuracy = accuracy_score(y_test, grid_predictions)
print("Accuracy:", accuracy)'''

'cm = confusion_matrix(y_test, grid_predictions)\nprint("Confusion Matrix:\n", cm)\nprint("\n")\ncf = classification_report(y_test, grid_predictions)\nprint("Classification Report:\n", cf)   \nprint("\n")\naccuracy = accuracy_score(y_test, grid_predictions)\nprint("Accuracy:", accuracy)'

In [282]:
from xgboost import XGBClassifier

In [283]:
XGB = XGBClassifier(n_estimators=1000,learning_rate=0.05,max_depth=6,subsample=0.8,colsample_bytree=0.8,objective='multi:softmax',num_class=2,eval_metric='mlogloss',tree_method='auto')


In [284]:
XGB.fit(X_combained_train_features, y_combained_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=None, num_class=2, ...)

In [285]:
frontal_predictions = XGB.predict(X_combained_test_features)

In [286]:
cm = confusion_matrix(y_combained_test, frontal_predictions)
print("Confusion Matrix:\n", cm)
print("\n")
cf = classification_report(y_combained_test, frontal_predictions)
print("Classification Report:\n", cf)
print("\n")
accuracy = accuracy_score(y_combained_test, frontal_predictions)
print("Accuracy:", accuracy)

Confusion Matrix:
 [[23 21]
 [12 29]]


Classification Report:
               precision    recall  f1-score   support

           0       0.66      0.52      0.58        44
           1       0.58      0.71      0.64        41

    accuracy                           0.61        85
   macro avg       0.62      0.62      0.61        85
weighted avg       0.62      0.61      0.61        85



Accuracy: 0.611764705882353
